# PID Command Test


In [1]:
# Import standard libraries
import csv
import math
from pathlib import Path
import time

# Import third-party libraries
from IPython.display import clear_output
import mujoco
import mujoco.viewer
import numpy as np

In [2]:
# Settings
MJCF_PATH = Path("/workspace/mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml")
MOTOR_SPEED_LIMIT = 1.0    # Max motor speed in each direction
PRINT_EVERY = 50           # Number of sim loop iterations before printing sensor readings

# Actuator names (from MJCF file)
LEFT_MOTOR = "left_motor"
RIGHT_MOTOR = "right_motor"

# Sensor names (from MJCF file)
IMU_ACCEL = "imu_accel"
IMU_GYRO = "imu_gyro"
IMU_ORIENTATION = "imu_orientation"
LEFT_WHEEL_POS = "left_wheel_pos"
RIGHT_WHEEL_POS = "right_wheel_pos"
LEFT_WHEEL_VEL = "left_wheel_vel"
RIGHT_WHEEL_VEL = "right_wheel_vel"

In [3]:
def clamp(x):
    """Limit motor speed and direction to a minimum and maximum"""
    return max(-MOTOR_SPEED_LIMIT, min(MOTOR_SPEED_LIMIT, x))

In [4]:
# Load model into MuJoCo
model = mujoco.MjModel.from_xml_path(str(MJCF_PATH))

# Use model to get the simulation state
data  = mujoco.MjData(model)

In [5]:
# Get ID of actuators from MJCF names
left_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, LEFT_MOTOR)
right_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, RIGHT_MOTOR)

# Print IDs
print(f"Left motor ID: {left_motor_id}")
print(f"Right motor ID: {right_motor_id}")

Left motor ID: 0
Right motor ID: 1


In [22]:
# Lean command
PITCH_OFFSET = 0.1

# Filter and PID coefficients (tune these)
ALPHA = 0.99
KP = 7.0      # Proportional constant
KD = 0.5      # Derivative constant

# When the robot has "tipped over" into an unrecoverable state
TIP_THRESHOLD = math.radians(30)

# Get the chassis ID from MJCF
chassis_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "chassis")

# Create a log file
log_file = open("pid_log.csv", "w", newline="")
log_writer = csv.writer(log_file)
log_writer.writerow(["step", "pitch", "pitch_rate", "accel_pitch", "motor_spd_target", "vel_actual"])

# Resets simulation data to defaults
mujoco.mj_resetData(model, data)
mujoco.mj_forward(model, data)

# Launch MuJoCo simulator and GUI
steps = 0
tipped = False
pitch = 0.0
prev_time = 0.0
try:
    with mujoco.viewer.launch_passive(model, data) as viewer:
        # Define free-look camera (control with mouse), looking at robot's back-right
        viewer.cam.type = mujoco.mjtCamera.mjCAMERA_FREE
        viewer.cam.lookat[:] = [0, 0, 0.05]
        viewer.cam.distance  = 0.8 
        viewer.cam.azimuth   = 45
        viewer.cam.elevation = -25
    
        # Simulation loop
        while viewer.is_running():
            step_start = time.time()
    
            # Detect MuJoCo reset and reset variables
            if data.time < prev_time:
                tipped = False
                mujoco.mj_forward(model, data)
                accel_x, _, accel_z = data.sensor(IMU_ACCEL).data
                pitch = -1 * math.atan2(accel_x, accel_z)
            prev_time = data.time
    
            # Read sensors
            accel_x, accel_y, accel_z = data.sensor(IMU_ACCEL).data
            pitch_rate = data.sensor(IMU_GYRO).data[1]
    
            # Accelerometer-derived pitch
            accel_pitch = -1 * math.atan2(accel_x, accel_z)
    
            # Copmlementary filter to estimate pitch from accelerometer and gyroscope
            pitch = ALPHA * (pitch + (pitch_rate * model.opt.timestep)) + \
                    (1 - ALPHA) * accel_pitch
    
            # Uncomment to use ground-truth pitch (from orientation) instead of estimated pitch
            # w, x, y, z = data.sensor(IMU_ORIENTATION).data
            # pitch = math.asin(max(-1.0, min(1.0, 2.0 * (w * y - z * x))))
    
            # Get body-frame forward velocity
            xmat = data.xmat[chassis_id].reshape(3, 3)
            world_vel = data.cvel[chassis_id][3:6]
            vel_actual = -float(np.dot(world_vel, xmat[:, 0]))
            
            # If the robot tips over, stop the wheels
            if abs(pitch) > TIP_THRESHOLD:
                tipped = True

            # Set target if not tipped
            if tipped:
                motor_spd_target = 0.0
            else:
                # PD controller
                motor_spd_target = clamp((KP * (pitch + PITCH_OFFSET)) + \
                                         (KD * pitch_rate))

            # Update motor speeds
            data.ctrl[left_motor_id] = motor_spd_target
            data.ctrl[right_motor_id] = motor_spd_target
    
            # Advance simulation by one step (determined by timestep attribute in MJCF file)
            mujoco.mj_step(model, data)
    
            # Render the current simulation state
            viewer.sync()
    
            # Just print what the sensors say when the robot is upright
            steps += 1
            if steps % 20 == 0:
                # Log to CSV file
                log_writer.writerow([steps, pitch, pitch_rate, accel_pitch, motor_spd_target, vel_actual])
                log_file.flush()
    
                # Print to output
                clear_output(wait=True)
                print(f"step:              {steps}")
                print(f"accel_pitch:        {accel_pitch:.4f} rad")
                print(f"pitch (filtered):   {pitch:.4f} rad")
                print(f"pitch_rate:         {pitch_rate:.4f} rad/s")
                print(f"vel_actual:         {vel_actual:.4f} m/s")
                # print(f"vel_error:          {vel_error:.4f}")
                # print(f"pitch_setpoint:     {pitch_setpoint:.4f} rad")
                print(f"motor_spd_target:   {motor_spd_target:.4f}")
                
            # Make sure the loop iteration time matches the MJCF timestep time (2 ms)
            # i.e. slow the simulation down so we can see the robot behave in real time
            if slack > 0:
                time.sleep(slack)
finally:
    log_file.close()
    print("Log file closed")

step:              720
accel_pitch:        3.1229 rad
pitch (filtered):   1.2958 rad
pitch_rate:         -0.2185 rad/s
vel_actual:         -0.0068 m/s
motor_spd_target:   0.0000
Log file closed


KeyboardInterrupt: 